In [ ]:
%pip install -r requirements.txt

In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix

# Download latest version
path = kagglehub.dataset_download("shauryasrivastava01/liver-patient-dataset")

print("Path to dataset files:", path)



In [ ]:
path = "Datasets/Teen_Mental_Health_Dataset.csv"
df = pd.read_csv(path)

In [ ]:
df

In [ ]:
df.head()

### Distribuzione della classe depression_label (il target)

In [ ]:
conteggi = df["depression_label"].value_counts()

plt.bar(conteggi.index, conteggi.values)
plt.xlabel("depression_label")
plt.ylabel("Frequenza")
plt.title('Distribuzione di "depression_label"')
plt.show()

### Creazione trainining e testing set

In [ ]:
feature_names = df.columns.drop("depression_label").tolist()

target_name = "depression_label"

categorical_features = set(["gender", "platform_usage", "social_interaction_level"])
continuos_features = set(feature_names) - categorical_features

print(categorical_features)
print(continuos_features)
feature_names = list(feature_names)

In [ ]:
df_encoded = df.copy()

# One-hot encoding: non introduce un ordine artificiale tra le categorie.
df_encoded = pd.get_dummies(
    df_encoded,
    columns=["gender", "platform_usage", "social_interaction_level"],
    dtype=int
)
df_encoded["depression_label"] = df_encoded["depression_label"].astype(int)
feature_names = df_encoded.columns.drop("depression_label").tolist()
categorical_features_encoded = [
    col for col in feature_names
    if col.startswith(("gender_", "platform_usage_", "social_interaction_level_"))
]
one_hot_groups = {
    base: [col for col in feature_names if col.startswith(f"{base}_")]
    for base in ["gender", "platform_usage", "social_interaction_level"]
}

# Schema esplicito: il tipo non viene dedotto dal numero di valori unici.
nominal_features = list(one_hot_groups)
ordinal_features = ["stress_level", "anxiety_level", "addiction_level"]
integer_features = ["age"]
continuous_features = [
    "daily_social_media_hours", "sleep_hours",
    "screen_time_before_sleep", "academic_performance",
    "physical_activity"
]
feature_bounds = {
    "age": (13, 19),
    "stress_level": (1, 10),
    "anxiety_level": (1, 10),
    "addiction_level": (1, 10),
    "daily_social_media_hours": (0, 24),
    "sleep_hours": (0, 24),
    "screen_time_before_sleep": (0, 24),
    "academic_performance": (0, 4),
    "physical_activity": (0, 24),
}

In [ ]:
df_encoded

In [ ]:
df_encoded["depression_label"].dtype

In [ ]:
df_encoded[categorical_features_encoded].dtypes

### addestriamo Il decisione Tree

In [ ]:
X = df_encoded[feature_names]
y = df_encoded[target_name]
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, random_state=42, stratify=y_train_val
)  # circa 70% train, 15% validation, 15% test

In [ ]:
df_encoded.head()

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:

model = DecisionTreeClassifier(random_state=42)
model = model.fit(X_train, y_train)

# Visualizza l'albero decisionale
fig, ax = plt.subplots(figsize=(150, 100))
plot_tree(model, filled=True, ax=ax)
plt.plot()

In [ ]:
# [
#     true 0, false 1
#     false 0, true 1
#  ]

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)
print("\nAccuracy:", cm.diagonal().sum() / cm.sum())

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

### Miglioriamo l'albero

In [ ]:
prunedModel = DecisionTreeClassifier(random_state=42,max_depth=5, min_samples_split=2)
prunedModel = prunedModel.fit(X_train, y_train)

# Visualizza l'albero decisionale
fig, ax = plt.subplots(figsize=(150, 100))
plot_tree(prunedModel, filled=True, ax=ax)
plt.plot()

In [ ]:
y_pred = prunedModel.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)
print("\nAccuracy:", cm.diagonal().sum() / cm.sum())

In [ ]:
#precision: Tra tutti quelli che il modello ha predetto come positivi, quanti erano davvero positivi?
#recall: Tra tutti quelli che erano davvero positivi, quanti ne ha trovati il modello?

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

### L'albero ha una bassa accuratezza, Usiamo il parametro di complessità

In [ ]:
path = model.cost_complexity_pruning_path(X_train, y_train)

ccp_alphas = path.ccp_alphas
print(ccp_alphas)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

results = []
#andare oltre 8 lo manda comunque in overfitting

for depth in range(4, 8):
    for complexity in ccp_alphas:
        clf = DecisionTreeClassifier(
            max_depth=depth,
            ccp_alpha=complexity,
            class_weight="balanced",
            random_state=42
        )

        clf.fit(X_train, y_train)

        train_acc = clf.score(X_train, y_train)
        validation_acc = clf.score(X_val, y_val)

        results.append({
            "max_depth": depth,
            "ccp_alpha": complexity,
            "train_accuracy": train_acc,
            "validation_accuracy": validation_acc
        })

In [ ]:
plt.figure(figsize=(10, 6))

for depth in range(4, 11):
    depth_results = [r for r in results if r["max_depth"] == depth]

    alphas = [r["ccp_alpha"] for r in depth_results]
    test_acc = [r["validation_accuracy"] for r in depth_results]

    plt.plot(alphas, test_acc, marker="o", label=f"max_depth={depth}")

plt.xlabel("Complexity Parameter ccp_alpha")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy vs ccp_alpha per diverse profondità")
plt.xscale("log")
plt.legend()
plt.show()

In [ ]:
best_result = max(results, key=lambda x: x["validation_accuracy"])

print("Miglior max_depth:", best_result["max_depth"])
print("Miglior ccp_alpha:", best_result["ccp_alpha"])
print("Train accuracy:", best_result["train_accuracy"])
print("Validation accuracy:", best_result["validation_accuracy"])

In [ ]:
plt.figure(figsize=(10, 6))

for depth in range(4, 11):
    depth_results = [r for r in results if r["max_depth"] == depth]

    alphas = [r["ccp_alpha"] for r in depth_results]
    train_acc = [r["train_accuracy"] for r in depth_results]

    plt.plot(alphas, train_acc, marker="o", label=f"max_depth={depth}")

plt.xlabel("Complexity Parameter ccp_alpha")
plt.ylabel("Training Accuracy")
plt.title("Training Accuracy vs ccp_alpha per diverse profondità")
plt.xscale("log")
plt.legend()
plt.show()

In [ ]:
best_clf = DecisionTreeClassifier(
    max_depth=best_result["max_depth"],
    ccp_alpha=best_result["ccp_alpha"],
    class_weight="balanced",
    random_state=42
)

best_clf.fit(X_train, y_train)

print("Accuracy train:", best_clf.score(X_train, y_train))
print("Accuracy test:", best_clf.score(X_test, y_test))

In [ ]:
y_pred = best_clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)
print("\nAccuracy:", cm.diagonal().sum() / cm.sum())

In [ ]:
print(classification_report(y_test, y_pred))

### diamo un occhiata con la balancedPrecision

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix

print(classification_report(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

### Andiamo a modificare l'addestramento per dare più peso ai dati minoritari

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


def train_best_weighted_tree(
    X_train,
    y_train,
    X_test,
    y_test,
    ccp_alphas,
    weights_to_try=None,
    depths=range(4, 8),
    scoring_metric="f1_class_1",
    pos_label=1,
    random_state=42,
    print_report=True
):
    """
    Addestra più DecisionTreeClassifier provando:
    - diverse profondità;
    - diversi valori di ccp_alpha;
    - diversi pesi manuali per le classi.

    Restituisce:
    - best_clf: modello migliore riaddestrato;
    - results_df: DataFrame con tutti i risultati;
    - best_row: riga del DataFrame corrispondente al miglior modello.
    """

    if weights_to_try is None:
        weights_to_try = [
            {0: 1, 1: 1},
            {0: 1, 1: 2},
            {0: 1, 1: 3},
            {0: 1, 1: 5},
            {0: 1, 1: 10}
        ]

    results = []

    for depth in depths:
        for complexity in ccp_alphas:
            for weights in weights_to_try:

                clf = DecisionTreeClassifier(
                    max_depth=depth,
                    ccp_alpha=complexity,
                    class_weight=weights,
                    random_state=random_state
                )

                clf.fit(X_train, y_train)

                y_pred_train = clf.predict(X_train)
                y_pred_test = clf.predict(X_test)

                train_acc = accuracy_score(y_train, y_pred_train)
                test_acc = accuracy_score(y_test, y_pred_test)

                train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
                test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)

                precision_1 = precision_score(
                    y_test,
                    y_pred_test,
                    pos_label=pos_label,
                    zero_division=0
                )

                recall_1 = recall_score(
                    y_test,
                    y_pred_test,
                    pos_label=pos_label,
                    zero_division=0
                )

                f1_1 = f1_score(
                    y_test,
                    y_pred_test,
                    pos_label=pos_label,
                    zero_division=0
                )

                macro_f1 = f1_score(
                    y_test,
                    y_pred_test,
                    average="macro",
                    zero_division=0
                )

                weighted_f1 = f1_score(
                    y_test,
                    y_pred_test,
                    average="weighted",
                    zero_division=0
                )

                results.append({
                    "max_depth": depth,
                    "ccp_alpha": complexity,
                    "class_weight": weights,
                    "weight_0": weights.get(0, np.nan),
                    "weight_1": weights.get(1, np.nan),

                    "train_accuracy": train_acc,
                    "test_accuracy": test_acc,

                    "train_balanced_accuracy": train_bal_acc,
                    "test_balanced_accuracy": test_bal_acc,

                    "precision_class_1": precision_1,
                    "recall_class_1": recall_1,
                    "f1_class_1": f1_1,

                    "macro_f1": macro_f1,
                    "weighted_f1": weighted_f1
                })

    results_df = pd.DataFrame(results)

    if scoring_metric not in results_df.columns:
        raise ValueError(
            f"scoring_metric non valido: {scoring_metric}. "
            f"Scegli tra: {list(results_df.columns)}"
        )

    best_index = results_df[scoring_metric].idxmax()
    best_row = results_df.loc[best_index]

    best_clf = DecisionTreeClassifier(
        max_depth=int(best_row["max_depth"]),
        ccp_alpha=best_row["ccp_alpha"],
        class_weight=best_row["class_weight"],
        random_state=random_state
    )

    best_clf.fit(X_train, y_train)

    y_pred_best = best_clf.predict(X_test)

    if print_report:
        print("Migliore configurazione trovata:")
        print(best_row)

        print("\nConfusion matrix:")
        print(confusion_matrix(y_test, y_pred_best))

        print("\nClassification report:")
        print(classification_report(y_test, y_pred_best, zero_division=0))

    return best_clf, results_df, best_row

In [ ]:
weights_to_try = [
    {0: 1, 1: 1},
    {0: 1, 1: 1.5},
    {0: 1, 1: 2},
    {0: 1, 1: 2.5},
    {0: 1, 1: 3},
    {0: 1, 1: 4},
    {0: 1, 1: 4.5},
    {0: 1, 1: 5},
    {0: 1, 1: 7},
    {0: 1, 1: 9},
    {0: 1, 1: 10}
]

best_clf, results_df, best_row = train_best_weighted_tree(
    X_train,
    y_train,
    X_val,
    y_val,
    ccp_alphas=ccp_alphas,
    weights_to_try=weights_to_try,
    depths=range(4, 8),
    scoring_metric="f1_class_1"
)


In [ ]:
y_pred_test = best_clf.predict(X_test)
print(classification_report(y_test, y_pred_test, zero_division=0))

### Facciamo un ensemble a majority

In [ ]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
def majority_vote(predictions):
    """
    predictions deve essere una lista con 3 array.

    Esempio:
    predictions = [
        predizioni_modello_1,
        predizioni_modello_2,
        predizioni_modello_3
    ]

    Ogni array contiene valori 0/1.
    """

    final_predictions = []

    n_samples = len(predictions[0])

    for i in range(n_samples):
        votes_for_1 = 0

        for model_predictions in predictions:
            if model_predictions[i] == 1:
                votes_for_1 = votes_for_1 + 1

        if votes_for_1 >= 2:
            final_predictions.append(1)
        else:
            final_predictions.append(0)

    return np.array(final_predictions)


def train_three_trees(
    X_train,
    y_train,
    max_depth,
    ccp_alpha,
    class_weight,
    random_state=42
):
    models = []

    splitter = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=random_state
    )

    fold_number = 0

    for train_index, part_index in splitter.split(X_train, y_train):
        X_part = X_train.iloc[train_index]
        y_part = y_train.iloc[train_index]

        tree = DecisionTreeClassifier(
            max_depth=max_depth,
            ccp_alpha=ccp_alpha,
            class_weight=class_weight,
            random_state=random_state + fold_number
        )

        tree.fit(X_part, y_part)

        models.append(tree)

        fold_number = fold_number + 1

    return models
def predict_ensemble(models, X_test):
    predictions = []

    for model in models:
        model_predictions = model.predict(X_test)
        predictions.append(model_predictions)

    final_predictions = majority_vote(predictions)

    return final_predictions

def search_best_ensemble_tree(
    X_train,
    y_train,
    X_test,
    y_test,
    ccp_alphas,
    weights_to_try,
    depths=range(4, 8),
    scoring_metric="f1_class_1",
    random_state=42
):
    results = []

    best_models = None
    best_score = -1
    best_row = None
    best_predictions = None

    for depth in depths:

        for alpha in ccp_alphas:

            for weights in weights_to_try:

                models = train_three_trees(
                    X_train=X_train,
                    y_train=y_train,
                    max_depth=depth,
                    ccp_alpha=alpha,
                    class_weight=weights,
                    random_state=random_state
                )

                y_pred = predict_ensemble(
                    models=models,
                    X_test=X_test
                )

                test_accuracy = accuracy_score(y_test, y_pred)

                test_balanced_accuracy = balanced_accuracy_score(
                    y_test,
                    y_pred
                )

                precision_class_1 = precision_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    zero_division=0
                )

                recall_class_1 = recall_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    zero_division=0
                )

                f1_class_1 = f1_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    zero_division=0
                )

                macro_f1 = f1_score(
                    y_test,
                    y_pred,
                    average="macro",
                    zero_division=0
                )

                weighted_f1 = f1_score(
                    y_test,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )

                row = {
                    "max_depth": depth,
                    "ccp_alpha": alpha,
                    "class_weight": weights,
                    "weight_0": weights[0],
                    "weight_1": weights[1],
                    "accuracy": test_accuracy,
                    "balanced_accuracy": test_balanced_accuracy,
                    "precision_class_1": precision_class_1,
                    "recall_class_1": recall_class_1,
                    "f1_class_1": f1_class_1,
                    "macro_f1": macro_f1,
                    "weighted_f1": weighted_f1
                }

                results.append(row)

                current_score = row[scoring_metric]

                if current_score > best_score:
                    best_score = current_score
                    best_models = models
                    best_row = row
                    best_predictions = y_pred

    results_df = pd.DataFrame(results)

    print("Migliore configurazione trovata:")
    print(best_row)

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, best_predictions))

    print("\nClassification report:")
    print(classification_report(y_test, best_predictions, zero_division=0))

    print("Metrica usata per scegliere:", scoring_metric)
    print("Valore migliore:", best_score)

    return best_models, results_df, best_row, best_predictions


In [ ]:
weights_to_try = [
    {0: 1, 1: 1},
    {0: 1, 1: 1.5},
    {0: 1, 1: 2},
    {0: 1, 1: 2.5},
    {0: 1, 1: 3},
    {0: 1, 1: 4},
    {0: 1, 1: 5}
]


#clean_training_results[] = best_models, results_df, best_row, best_predictions

best_models, results_df, best_row, best_predictions = search_best_ensemble_tree(
    X_train=X_train,
    y_train=y_train,
    X_test=X_val,  # validation set usato per la selezione
    y_test=y_val,
    ccp_alphas=ccp_alphas,
    weights_to_try=weights_to_try,
    depths=range(4, 8),
    scoring_metric="f1_class_1",
    random_state=42
)
clean_training_results = best_models, results_df, best_row, best_predictions

In [ ]:
results_df.sort_values(
    by="f1_class_1",
    ascending=False
).head(10)

In [ ]:
y_pred = predict_ensemble(
                    models=best_models,
                    X_test=X_test
                )
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
n_estimators_to_try = [100, 200, 300, 500]

depths_to_try = [4, 5, 6, 7, 8, None]

weights_to_try = [
    {0: 1, 1: 1},
    {0: 1, 1: 1.5},
    {0: 1, 1: 2},
    {0: 1, 1: 3},
    {0: 1, 1: 5},
    "balanced"
]

# best_rf, rf_results_df, best_rf_row, y_pred_rf = search_best_random_forest(
#     X_train,
#     y_train,
#     X_test,
#     y_test,
#     n_estimators_to_try=n_estimators_to_try,
#     depths_to_try=depths_to_try,
#     weights_to_try=weights_to_try,
#     scoring_metric="macro_f1"
# )

### Sporchiamo Il Dataset

In [ ]:
# funzione per sporcare variabili categoriche
import numpy as np

import numpy as np







def corrupt_rows_cat(X_train, col, frac=0.1, random_state=None):
    X_train_corrupt = X_train.copy()

    # Generatore casuale riproducibile
    rng = np.random.default_rng(random_state)

    # Recupero i valori univoci della colonna
    unique_values = X_train_corrupt[col].unique()

    # Se esiste un solo valore, non posso sostituirlo con qualcosa di diverso
    if len(unique_values) <= 1:
        print(f"Attenzione: la colonna '{col}' ha un solo valore unico. Nessuna modifica applicata.")
        return X_train_corrupt

    # Numero di righe da corrompere
    n_rows_to_corrupt = int(frac * len(X_train_corrupt))

    # Scelgo righe casuali senza ripetizione
    indices_to_corrupt = rng.choice(
        X_train_corrupt.index,
        size=n_rows_to_corrupt,
        replace=False
    )

    # Per ogni riga scelta, sostituisco il valore con un altro diverso
    for idx in indices_to_corrupt:
        current_value = X_train_corrupt.at[idx, col]

        pool_of_new_values = unique_values[unique_values != current_value]

        X_train_corrupt.at[idx, col] = rng.choice(pool_of_new_values)

    return X_train_corrupt


def corrupt_rows_cont(
    X_train, col, frac=0.1, noise_level=0.5, random_state=None,
    lower_bound=None, upper_bound=None
):
    X_train_corrupt = X_train.copy()

    rng = np.random.default_rng(random_state)

    # Converto la colonna in float, così posso inserirci valori decimali
    X_train_corrupt[col] = X_train_corrupt[col].astype(float)

    # Numero di righe da corrompere
    n_rows_to_corrupt = int(frac * len(X_train_corrupt))

    # Scelgo gli indici delle righe da corrompere
    indices_to_corrupt = rng.choice(
        X_train_corrupt.index,
        size=n_rows_to_corrupt,
        replace=False
    )

    # Deviazione standard della colonna
    std_col = X_train_corrupt[col].std()

    if std_col == 0:
        print(f"Attenzione: la colonna '{col}' ha deviazione standard 0. Nessuna modifica applicata.")
        return X_train_corrupt

    for idx in indices_to_corrupt:
        current_value = X_train_corrupt.at[idx, col]

        noise = rng.normal(
            loc=0,
            scale=noise_level * std_col
        )

        new_value = current_value + noise
        if lower_bound is not None or upper_bound is not None:
            new_value = np.clip(new_value, lower_bound, upper_bound)
        X_train_corrupt.at[idx, col] = new_value

    return X_train_corrupt


def corrupt_rows_ordinal(
    X_train, col, frac=0.1, random_state=None, lower_bound=None, upper_bound=None
):
    """Applica una variazione intera locale e rispetta i limiti del dominio."""
    X_corrupt = X_train.copy()
    rng = np.random.default_rng(random_state)
    n_rows = int(frac * len(X_corrupt))
    indices = rng.choice(X_corrupt.index, size=n_rows, replace=False)
    for idx in indices:
        current = int(X_corrupt.at[idx, col])
        alternatives = [
            current + delta for delta in (-2, -1, 1, 2)
            if lower_bound <= current + delta <= upper_bound
        ]
        X_corrupt.at[idx, col] = int(rng.choice(alternatives))
    return X_corrupt


def feature_type(col):
    group_name, _ = one_hot_group_for_column(col)
    if group_name is not None:
        return "nominale"
    if col in ordinal_features:
        return "ordinale"
    if col in integer_features:
        return "intera"
    if col in continuous_features:
        return "continua"
    raise ValueError(f"Feature senza tipo esplicito: {col}")


def one_hot_group_for_column(col):
    for group_name, group_cols in one_hot_groups.items():
        if col in group_cols:
            return group_name, group_cols
    return None, None


def corrupt_rows_one_hot(X_train, group_cols, frac=0.1, random_state=None):
    """Cambia categoria mantenendo esattamente una dummy attiva."""
    X_corrupt = X_train.copy()
    rng = np.random.default_rng(random_state)
    n_rows = int(frac * len(X_corrupt))
    if n_rows == 0 or len(group_cols) <= 1:
        return X_corrupt

    indices = rng.choice(X_corrupt.index, size=n_rows, replace=False)
    for idx in indices:
        active = [col for col in group_cols if X_corrupt.at[idx, col] == 1]
        if len(active) != 1:
            raise ValueError(f"One-hot non valido alla riga {idx}: {group_cols}")
        alternatives = [col for col in group_cols if col != active[0]]
        new_active = rng.choice(alternatives)
        X_corrupt.loc[idx, group_cols] = 0
        X_corrupt.at[idx, new_active] = 1
    return X_corrupt


def expand_feature_groups(features):
    expanded = []
    for feature in features:
        expanded.extend(one_hot_groups.get(feature, [feature]))
    return list(dict.fromkeys(expanded))


def corrupt_feature_columns(
    X_train, cols, frac, noise_level, random_state, categorical_cols
):
    """Corrompe feature numeriche o interi gruppi one-hot una sola volta."""
    X_corrupt = X_train.copy()
    processed_groups = set()
    for col_index, col in enumerate(cols):
        seed = random_state + col_index
        group_name, group_cols = one_hot_group_for_column(col)
        if group_name is not None:
            if group_name in processed_groups:
                continue
            X_corrupt = corrupt_rows_one_hot(
                X_corrupt, group_cols, frac=frac, random_state=seed
            )
            processed_groups.add(group_name)
        elif col in categorical_cols:
            X_corrupt = corrupt_rows_cat(
                X_corrupt, col=col, frac=frac, random_state=seed
            )
        elif col in ordinal_features or col in integer_features:
            lower, upper = feature_bounds[col]
            X_corrupt = corrupt_rows_ordinal(
                X_corrupt, col=col, frac=frac, random_state=seed,
                lower_bound=lower, upper_bound=upper
            )
        else:
            if col not in continuous_features:
                raise ValueError(f"Strategia di rumore non definita per: {col}")
            lower, upper = feature_bounds[col]
            X_corrupt = corrupt_rows_cont(
                X_corrupt, col=col, frac=frac, noise_level=noise_level,
                random_state=seed, lower_bound=lower, upper_bound=upper
            )
    return X_corrupt



### mostro le variabili dalla più importante alla meno importante

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

def grouped_mutual_information(X, y, categorical_cols, random_state=42):
    # Ricostruisce una sola variabile discreta per ciascun gruppo one-hot.
    grouped_data = {}
    discrete_mask = []
    grouped_columns = set()
    for group_name, group_cols in one_hot_groups.items():
        present = [col for col in group_cols if col in X.columns]
        if not present:
            continue
        if not (X[present].sum(axis=1) == 1).all():
            raise ValueError(f"Gruppo one-hot non valido: {group_name}")
        grouped_data[group_name] = np.argmax(X[present].to_numpy(), axis=1)
        discrete_mask.append(True)
        grouped_columns.update(present)
    for col in X.columns:
        if col not in grouped_columns:
            grouped_data[col] = X[col].to_numpy()
            discrete_mask.append(col in categorical_cols)
    grouped_X = pd.DataFrame(grouped_data, index=X.index)
    values = mutual_info_classif(
        grouped_X, y, discrete_features=discrete_mask, random_state=random_state
    )
    return pd.DataFrame({
        "variabile": grouped_X.columns, "importanza": values
    }).sort_values("importanza", ascending=False)


def select_feature_group(X, y, categorical_cols, feature_group, random_state=42):
    ranking = grouped_mutual_information(X, y, categorical_cols, random_state)
    ordered = ranking["variabile"].tolist()
    middle = len(ordered) // 2
    selected = ordered[:middle] if feature_group == "migliori" else ordered[middle:]
    return expand_feature_groups(selected), ranking


# Importanza stimata esclusivamente sul training e aggregata per feature originale.
importanza_variabili = grouped_mutual_information(
    X_train, y_train, categorical_features_encoded, random_state=42
)
print(importanza_variabili)

In [ ]:
feature_ordinate = importanza_variabili["variabile"].tolist()
n = len(feature_ordinate)
meta = n // 2

if n % 2 == 0:
    migliori = feature_ordinate[:meta]
    peggiori = feature_ordinate[meta:]
else:
    # la feature centrale appartiene a entrambe le liste
    migliori = feature_ordinate[:meta + 1]
    peggiori = feature_ordinate[meta:]

print("Migliori:", migliori)
print("Peggiori:", peggiori)

In [ ]:
import matplotlib.pyplot as plt

importanza_variabili.plot(
    kind="barh",
    x="variabile",
    y="importanza",
    figsize=(8, 6)
)

plt.gca().invert_yaxis()
plt.show()

### Proviamo a sporcare la feature SGPT in maniera progressiva mentre lo addestriamo e vediamo di quanto cala la performance del modello

In [ ]:
df_encoded.head()

In [ ]:
## AG ratio è continua, voglio quindi usare la funzione che sporca colonno continue
# una volta sporcata faccio partire un giro di training su ensemble
# dopo di che faccio una valutazione del modello e la confronto con il modello addestrato sul dataset non sporco

In [ ]:
non_corrupted_data = [X_train, X_test, y_train, y_test]

In [ ]:
# df_corrotto = corrupt_rows_cont(
#     df_encoded,
#     col="Sgpt",
#     frac=0.2,
#     noise_level=0.5,
#     random_state=42
# )

In [ ]:
# Confronto controllato: stesso split del modello pulito, ma training corrotto.
cols_noise_comparison = expand_feature_groups(migliori)
X_train_noisy = corrupt_feature_columns(
    X_train, cols_noise_comparison, frac=0.2, noise_level=0.5,
    random_state=42, categorical_cols=categorical_features_encoded
)

noise_training_results = search_best_ensemble_tree(
    X_train=X_train_noisy,
    y_train=y_train,
    X_test=X_val,  # validation set usato per la selezione
    y_test=y_val,
    ccp_alphas=ccp_alphas,
    weights_to_try=weights_to_try,
    depths=range(4, 8),
    scoring_metric="f1_class_1",
    random_state=42
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Risultati modello pulito
clean_best_models, clean_results_df, clean_best_row, clean_best_predictions = clean_training_results

# Risultati modello con rumore
noise_best_models, noise_results_df, noise_best_row, noise_best_predictions = noise_training_results


# Funzione per trasformare best_row sempre in dizionario
def best_row_to_dict(best_row):
    if isinstance(best_row, pd.DataFrame):
        return best_row.iloc[0].to_dict()
    
    if isinstance(best_row, pd.Series):
        return best_row.to_dict()
    
    if isinstance(best_row, dict):
        return best_row
    
    raise TypeError("best_row deve essere un dict, una Series o un DataFrame")


clean_best_row = best_row_to_dict(clean_best_row)
noise_best_row = best_row_to_dict(noise_best_row)


# Metriche che voglio confrontare
possible_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision_class_1",
    "recall_class_1",
    "f1_class_1"
]

# Tengo solo le metriche presenti in entrambi
metrics = []

for metric in possible_metrics:
    if metric in clean_best_row.keys() and metric in noise_best_row.keys():
        metrics.append(metric)


# Valori delle metriche
clean_values = []

for metric in metrics:
    clean_values.append(clean_best_row[metric])


noise_values = []

for metric in metrics:
    noise_values.append(noise_best_row[metric])


# Grafico
x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(x - width/2, clean_values, width, label="Dataset pulito")
plt.bar(x + width/2, noise_values, width, label="Dataset con rumore")

plt.xticks(x, metrics, rotation=30)
plt.ylabel("Valore metrica")
plt.title("Confronto modello addestrato su dati puliti vs dati rumorosi")
plt.ylim(0, 1)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
df_encoded.head()

In [ ]:

# X_train, X_test, y_train, y_test = train_test_split_with_undersampling(
#     df=df_encoded,
#     target_col="Gender",
#     test_size=0.7,
#     random_state=42,
#     stratify=False
# )



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier


def train_three_trees_noise(
    X_train, y_train, max_depth, ccp_alpha, class_weight, random_state=42
):
    """Versione corretta usata soltanto dagli esperimenti sul rumore."""
    models = []
    splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)

    for fold_number, (train_index, _) in enumerate(splitter.split(X_train, y_train)):
        tree = DecisionTreeClassifier(
            max_depth=max_depth,
            ccp_alpha=ccp_alpha,
            class_weight=class_weight,
            random_state=random_state + fold_number
        )
        tree.fit(X_train.iloc[train_index], y_train.iloc[train_index])
        models.append(tree)

    return models


def search_best_ensemble_tree_noise(*args, **kwargs):
    original_train_function = globals()["train_three_trees"]
    globals()["train_three_trees"] = train_three_trees_noise
    try:
        return search_best_ensemble_tree(*args, **kwargs)
    finally:
        globals()["train_three_trees"] = original_train_function


# La classe 1 è minoritaria in questo dataset.
tree_weights_to_try = [
    {0: 1, 1: 1},
    {0: 1, 1: 1.5},
    {0: 1, 1: 2},
    {0: 1, 1: 2.5},
    {0: 1, 1: 3},
    {0: 1, 1: 4},
    {0: 1, 1: 5}
]


def test_noise_progressivo(
    df,
    target_col,
    cols_to_corrupt,
    ccp_alphas,
    weights_to_try,
    depths=range(4, 8),
    noise_level=0.5,
    test_size=0.3,
    validation_size=0.2,
    random_state=42,
    scoring_metric="f1_class_1",
    feature_group=None,
    stratify=False,
    categorical_cols=None,
    max_unique_for_cat=10
):

    risultati_finali = []
    risultati_completi = {}

    if categorical_cols is None:
        categorical_cols = []

    if isinstance(cols_to_corrupt, str):
        cols_to_corrupt = [cols_to_corrupt]

    for c in cols_to_corrupt:
        if c == target_col:
            raise ValueError("Non devi sporcare la colonna target.")

    # Split train/validation/test. Il test non partecipa alla selezione.
    X = df.drop(columns=[target_col])
    y = df[target_col].astype(int)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y if stratify else None
    )

    X_train_full, X_val, y_train_full, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=validation_size,
        random_state=random_state,
        stratify=y_train_val if stratify else None
    )

    # La Mutual Information viene calcolata esclusivamente sul training set.
    if feature_group is not None:
        if feature_group not in {"migliori", "peggiori"}:
            raise ValueError("feature_group deve essere 'migliori' o 'peggiori'")

        cols_to_corrupt, feature_ranking = select_feature_group(
            X_train_full, y_train_full, categorical_cols,
            feature_group, random_state
        )

    tipo_colonne = {col: feature_type(col) for col in cols_to_corrupt}

    print("Colonne da corrompere:")
    for col, tipo in tipo_colonne.items():
        print(f" - {col}: {tipo}")

    # ✔ rumore progressivo (include 0%)
    percentuali_rumore = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]

    for frac in percentuali_rumore:

        print(f"\nRumore: {int(frac*100)}%")

        # reset train base
        X_train = X_train_full.copy()
        y_train = y_train_full.copy()

        X_train = corrupt_feature_columns(
            X_train, cols_to_corrupt, frac, noise_level, random_state,
            categorical_cols
        )

        # ✔ training modello (ensemble tree search invariato)
        training_results = search_best_ensemble_tree_noise(
            X_train=X_train,
            y_train=y_train,
            # La configurazione viene scelta sul validation set.
            X_test=X_val,
            y_test=y_val,
            ccp_alphas=ccp_alphas,
            weights_to_try=weights_to_try,
            depths=depths,
            scoring_metric=scoring_metric,
            random_state=random_state
        )

        best_models, results_df, best_row, validation_predictions = training_results

        # Il test set viene usato soltanto dopo la selezione dei parametri.
        best_predictions = predict_ensemble(best_models, X_test)
        accuracy = accuracy_score(y_test, best_predictions)
        balanced_accuracy = balanced_accuracy_score(y_test, best_predictions)
        precision = precision_score(y_test, best_predictions, pos_label=1, zero_division=0)
        recall = recall_score(y_test, best_predictions, pos_label=1, zero_division=0)
        f1 = f1_score(y_test, best_predictions, pos_label=1, zero_division=0)

        # ✔ normalizzazione best_row
        if isinstance(best_row, pd.DataFrame):
            best_row_dict = best_row.iloc[0].to_dict()
        elif isinstance(best_row, pd.Series):
            best_row_dict = best_row.to_dict()
        elif isinstance(best_row, dict):
            best_row_dict = best_row.copy()
        else:
            raise TypeError("best_row formato non valido")

        best_row_dict.update({
            "accuracy": accuracy,
            "balanced_accuracy": balanced_accuracy,
            "precision_class_1": precision,
            "recall_class_1": recall,
            "f1_class_1": f1,
            "colonne_sporcate": ",".join(cols_to_corrupt),
            "tipo_colonne": str(tipo_colonne),
            "frac_rumore": frac,
            "percentuale_rumore": int(frac * 100)
        })

        risultati_finali.append(best_row_dict)

        risultati_completi[int(frac * 100)] = {
            "X_train": X_train,
            "X_val": X_val,
            "X_test": X_test,
            "y_train": y_train,
            "y_val": y_val,
            "y_test": y_test,
            "best_models": best_models,
            "results_df": results_df,
            "best_row": best_row,
            "best_predictions": best_predictions,
            "accuracy": accuracy
        }

    # ✔ dataframe finale
    risultati_df = pd.DataFrame(risultati_finali)

    # ✔ metriche disponibili
    metriche_possibili = [
        "accuracy",
        "balanced_accuracy",
        "precision_class_1",
        "recall_class_1",
        "f1_class_1"
    ]

    metriche_presenti = [
        m for m in metriche_possibili if m in risultati_df.columns
    ]

    print(risultati_df)

    # ✔ grafico
    plt.figure(figsize=(10, 6))

    for m in metriche_presenti:

        plt.plot(
            risultati_df["percentuale_rumore"],
            risultati_df[m],
            marker="o",
            label=m
        )

    plt.xlabel("Percentuale di rumore")
    plt.ylabel("Metriche")
    plt.title(f"Robustezza ensemble alberi su {cols_to_corrupt}")
    plt.ylim(0, 1)
    plt.xticks(risultati_df["percentuale_rumore"])
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    return risultati_df, risultati_completi

In [ ]:
df_encoded.head()

In [ ]:
risultati_rumore_df, risultati_completi_rumore = test_noise_progressivo(
    df=df_encoded,
    target_col="depression_label",
    cols_to_corrupt=peggiori,
    ccp_alphas=ccp_alphas,
    weights_to_try=tree_weights_to_try,
    depths=range(4, 8),
    noise_level=0.5,
    test_size=0.3,
    random_state=42,
    scoring_metric="f1_class_1",
    feature_group="peggiori",
    categorical_cols=categorical_features_encoded,
    stratify=True
)


In [ ]:
risultati_rumore_df, risultati_completi_rumore = test_noise_progressivo(
    df=df_encoded,
    target_col="depression_label",
    cols_to_corrupt=migliori,
    ccp_alphas=ccp_alphas,
    weights_to_try=tree_weights_to_try,
    depths=range(4, 8),
    noise_level=0.5,
    test_size=0.3,
    random_state=42,
    scoring_metric="f1_class_1",
    feature_group="migliori",
    categorical_cols=categorical_features_encoded,
    stratify=True
)

Inizio rete neurale


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import tensorflow as tf


In [ ]:
y = df_encoded["depression_label"]
X_train, X_test, y_train, y_test = train_test_split(df_encoded[feature_names], df_encoded[target_name], test_size=0.3, random_state=42, stratify=y)

In [ ]:
scaler = RobustScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_pred = model.predict(X_test)

# conversione probabilità in classi
y_pred = (y_pred > 0.5).astype(int)



cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)

accuracy = cm.diagonal().sum() / cm.sum()
print("\nAccuracy:", accuracy)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score
)


def build_model_NN(input_dim, seed=None):
    tf.keras.backend.clear_session()

    if seed is not None:
        tf.keras.utils.set_random_seed(seed)

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation="relu", input_shape=(input_dim,)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy"
    )

    return model


def ensemble_predict_NN(models, X):
    preds = []

    for m in models:
        preds.append(m.predict(X, verbose=0))

    mean_pred = np.mean(preds, axis=0)

    return (mean_pred > 0.5).astype(int).flatten()


def test_noise_progressivo_NN(
    df,
    target_col,
    cols_to_corrupt,
    noise_level=0.5,
    test_size=0.3,
    validation_size=0.2,
    random_state=42,
    stratify=False,
    categorical_cols=None,
    max_unique_for_cat=10,
    feature_group=None,
    n_repeats=5,
    ensemble_size=3
):
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.utils.class_weight import compute_class_weight

    risultati_finali = []
    risultati_completi = {}

    if categorical_cols is None:
        categorical_cols = []

    if isinstance(cols_to_corrupt, str):
        cols_to_corrupt = [cols_to_corrupt]

    if target_col in cols_to_corrupt:
        raise ValueError("Non puoi sporcare il target")

    X = df.drop(columns=[target_col])

    # Il target è già codificato correttamente come 0/1.
    y = df[target_col].astype(int)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y if stratify else None
    )

    X_train_full, X_val, y_train_full, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=validation_size,
        random_state=random_state,
        stratify=y_train_val if stratify else None
    )

    # I gruppi di feature vengono determinati usando soltanto il training set.
    if feature_group is not None:
        if feature_group not in {"migliori", "peggiori"}:
            raise ValueError("feature_group deve essere 'migliori' o 'peggiori'")

        cols_to_corrupt, feature_ranking = select_feature_group(
            X_train_full, y_train_full, categorical_cols,
            feature_group, random_state
        )

    tipo_colonne = {col: feature_type(col) for col in cols_to_corrupt}

    print("Colonne da corrompere:")
    for col, tipo in tipo_colonne.items():
        print(f" - {col}: {tipo}")

    classes = np.sort(y_train_full.unique())
    balanced_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train_full
    )
    class_weights = dict(zip(classes, balanced_weights))
    print("Class weights:", class_weights)

    percentuali_rumore = [
        0, 0.1, 0.2, 0.3, 0.4, 0.5,
        0.6, 0.7, 0.8, 0.9, 1.0
    ]
    metric_names = [
        "accuracy",
        "balanced_accuracy",
        "precision_class_1",
        "recall_class_1",
        "f1_class_1"
    ]

    for frac in percentuali_rumore:
        percentuale = int(frac * 100)
        print(f"\nRumore: {percentuale}%")
        repeat_metrics = []
        repeat_details = []

        for repeat in range(n_repeats):
            X_train = X_train_full.copy()
            y_train = y_train_full.copy()

            X_train = corrupt_feature_columns(
                X_train, cols_to_corrupt, frac, noise_level,
                random_state + repeat * 1000, categorical_cols
            )

            scaler = RobustScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_val_scaled = scaler.transform(X_val)
            X_test_scaled = scaler.transform(X_test)

            models = []
            model_seeds = []

            for model_index in range(ensemble_size):
                # Lo stesso repeat/model usa bootstrap e inizializzazione uguali
                # per tutti i livelli: cambia soltanto il rumore nei dati.
                seed_i = random_state + repeat * 1000 + model_index
                model_seeds.append(seed_i)
                rng_bootstrap = np.random.default_rng(seed_i)
                idx = rng_bootstrap.choice(
                    len(X_train_scaled),
                    len(X_train_scaled),
                    replace=True
                )

                X_boot = X_train_scaled[idx]
                y_boot = y_train.iloc[idx].to_numpy()
                model = build_model_NN(
                    input_dim=X_train_scaled.shape[1],
                    seed=seed_i
                )
                model.fit(
                    X_boot,
                    y_boot,
                    epochs=30,
                    batch_size=32,
                    validation_data=(X_val_scaled, y_val),
                    class_weight=class_weights,
                    callbacks=[tf.keras.callbacks.EarlyStopping(
                        monitor="val_loss",
                        patience=5,
                        restore_best_weights=True
                    )],
                    verbose=0,
                    shuffle=True
                )
                models.append(model)

            y_pred = ensemble_predict_NN(models, X_test_scaled)
            metrics = {
                "accuracy": accuracy_score(y_test, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
                "precision_class_1": precision_score(
                    y_test, y_pred, pos_label=1, zero_division=0
                ),
                "recall_class_1": recall_score(
                    y_test, y_pred, pos_label=1, zero_division=0
                ),
                "f1_class_1": f1_score(
                    y_test, y_pred, pos_label=1, zero_division=0
                )
            }
            repeat_metrics.append(metrics)
            repeat_details.append({
                "repeat": repeat,
                "model_seeds": model_seeds,
                "y_pred": y_pred,
                "metrics": metrics
            })

            # Non conserviamo i modelli Keras di tutti i repeat in memoria.
            del models

        result = {
            "colonne_sporcate": ",".join(cols_to_corrupt),
            "frac_rumore": frac,
            "percentuale_rumore": percentuale,
            "n_repeats": n_repeats
        }
        for metric in metric_names:
            values = np.array([row[metric] for row in repeat_metrics])
            result[metric] = values.mean()
            result[f"{metric}_std"] = values.std(ddof=1) if n_repeats > 1 else 0.0

        risultati_finali.append(result)
        risultati_completi[percentuale] = {
            "X_train_clean": X_train_full,
            "X_val": X_val,
            "X_test": X_test,
            "y_train": y_train_full,
            "y_val": y_val,
            "y_test": y_test,
            "class_weights": class_weights,
            "repeats": repeat_details
        }

    risultati_df = pd.DataFrame(risultati_finali)

    plt.figure(figsize=(10, 6))
    for metric in metric_names:
        plt.errorbar(
            risultati_df["percentuale_rumore"],
            risultati_df[metric],
            yerr=risultati_df[f"{metric}_std"],
            marker="o",
            capsize=3,
            label=metric
        )

    plt.xlabel("Percentuale rumore")
    plt.ylabel("Metriche (media ± deviazione standard)")
    plt.title(f"Robustezza NN ensemble su {cols_to_corrupt}")
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print(risultati_df)
    return risultati_df, risultati_completi


In [ ]:
risultati_rumore_df, risultati_completi_rumore = test_noise_progressivo_NN(
    df=df_encoded,
    target_col="depression_label",
    cols_to_corrupt=peggiori,
    noise_level=0.5,
    test_size=0.3,
    random_state=42,
    feature_group="peggiori",
    n_repeats=5,
    categorical_cols=categorical_features_encoded,
    stratify=True,
)

In [ ]:
risultati_rumore_df, risultati_completi_rumore = test_noise_progressivo_NN(
    df=df_encoded,
    target_col="depression_label",
    cols_to_corrupt=migliori,
    noise_level=0.5,
    test_size=0.3,
    random_state=42,
    feature_group="migliori",
    n_repeats=5,
    categorical_cols=categorical_features_encoded,
    stratify=True,
)

Inizio svg

In [ ]:
y = df_encoded["depression_label"]
X_train, X_test, y_train, y_test = train_test_split(df_encoded[feature_names], df_encoded[target_name], test_size=0.3, random_state=42, stratify=y)

In [ ]:
scaler = RobustScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

# Creazione del modello SVM
model = SVC(
    kernel="rbf",        
    C=1.0,
    gamma="scale",
    class_weight="balanced"
)

# Addestramento
model.fit(X_train, y_train)

# Predizioni
y_pred = model.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)

accuracy = cm.diagonal().sum() / cm.sum()
print("\nAccuracy:", accuracy)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score
)


def build_model_SVM():
    model = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    )
    return model


def ensemble_predict_SVM(models, X):

    preds = []

    for m in models:
        preds.append(m.predict(X))

    preds = np.array(preds)

    # Majority voting
    y_pred = (np.mean(preds, axis=0) >= 0.5).astype(int)

    return y_pred




def test_noise_progressivo_SVM(
    df,
    target_col,
    cols_to_corrupt,
    noise_level=0.1,
    test_size=0.3,
    random_state=42,
    stratify=False,
    feature_group=None,
    categorical_cols=None,
    max_unique_for_cat=10
):

    risultati_finali = []
    risultati_completi = {}

    if categorical_cols is None:
        categorical_cols = []

    if isinstance(cols_to_corrupt, str):
        cols_to_corrupt = [cols_to_corrupt]

    for c in cols_to_corrupt:
        if c == target_col:
            raise ValueError("Non puoi sporcare il target")

    tipo_colonne = {col: feature_type(col) for col in cols_to_corrupt}

    # SPLIT UNA SOLA VOLTA (FIX come NN)
    X = df.drop(columns=[target_col])
    # Il target è già codificato correttamente come 0/1.
    y = df[target_col].astype(int)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y if stratify else None
    )

    # La selezione delle feature usa soltanto il training set.
    if feature_group is not None:
        if feature_group not in {"migliori", "peggiori"}:
            raise ValueError("feature_group deve essere 'migliori' o 'peggiori'")

        cols_to_corrupt, feature_ranking = select_feature_group(
            X_train_full, y_train_full, categorical_cols,
            feature_group, random_state
        )
        tipo_colonne = {col: feature_type(col) for col in cols_to_corrupt}

    print("Colonne da corrompere:")
    for col, tipo in tipo_colonne.items():
        print(f" - {col}: {tipo}")

    percentuali_rumore = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]

    for frac in percentuali_rumore:

        print(f"\nRumore: {int(frac*100)}%")

        X_train = X_train_full.copy()
        y_train = y_train_full.copy()

        X_train = corrupt_feature_columns(
            X_train, cols_to_corrupt, frac, noise_level, random_state,
            categorical_cols
        )

        # Scaling stimato esclusivamente sul training del livello corrente.
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        models = []

        # ENSEMBLE CON BOOTSTRAP (FIX come NN)
        for i in range(3):


            # quesra riga non va bene, non ha un seed
            #idx = np.random.choice(len(X_train), len(X_train), replace=True)
            #fix:
            seed_i = random_state + i
            rng = np.random.default_rng(seed_i)

            idx = rng.choice(
                len(X_train_scaled),
                len(X_train_scaled),
                replace=True
            )

            X_boot = X_train_scaled[idx]
            y_boot = y_train.iloc[idx].to_numpy()

            model = SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced"
            )

            model.fit(X_boot, y_boot)

            models.append(model)

        # PREDIZIONE ENSEMBLE (majority vote coerente)
        preds = []

        for m in models:
            preds.append(m.predict(X_test_scaled))

        preds = np.array(preds)

        y_pred = (np.mean(preds, axis=0) >= 0.5).astype(int)

        # METRICHE
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        balanced_acc = balanced_accuracy_score(y_test, y_pred)

        risultati_finali.append({
            "accuracy": accuracy,
            "f1_class_1": f1,
            "precision_class_1": precision,
            "recall_class_1": recall,
            "balanced_accuracy": balanced_acc,
            "colonne_sporcate": ",".join(cols_to_corrupt),
            "frac_rumore": frac,
            "percentuale_rumore": int(frac * 100)
        })

        risultati_completi[int(frac * 100)] = {
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
            "models": models,
            "scaler": scaler,
            "y_pred": y_pred
        }

    risultati_df = pd.DataFrame(risultati_finali)

    # GRAFICO IDENTICO NN
    metriche = [
        "accuracy",
        "balanced_accuracy",
        "precision_class_1",
        "recall_class_1",
        "f1_class_1"
    ]

    plt.figure(figsize=(10, 6))

    for m in metriche:
        plt.plot(
            risultati_df["percentuale_rumore"],
            risultati_df[m],
            marker="o",
            label=m
        )

    plt.xlabel("Percentuale rumore")
    plt.ylabel("Metriche")
    plt.title(f"Robustezza SVM ensemble su {cols_to_corrupt}")
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print(risultati_df)

    return risultati_df, risultati_completi

In [ ]:
risultati_rumore_df, risultati_completi_rumore = test_noise_progressivo_SVM(
    df=df_encoded,
    target_col="depression_label",
    cols_to_corrupt=peggiori,
    noise_level=0.5,
    test_size=0.3,
    random_state=42,
    feature_group="peggiori",
    categorical_cols=categorical_features_encoded,
    stratify=True,
)

In [ ]:
risultati_rumore_df, risultati_completi_rumore = test_noise_progressivo_SVM(
    df=df_encoded,
    target_col="depression_label",
    cols_to_corrupt=migliori,
    noise_level=0.5,
    test_size=0.3,
    random_state=42,
    feature_group="migliori",
    categorical_cols=categorical_features_encoded,
    stratify=True,
)